<a href="https://colab.research.google.com/github/vidaldominguez/uimp_deep_learning/blob/main/labs/P2_Aprendizaje_por_transferencia_en_CNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Práctica 2: Aprendizaje por transferencia en CNNs**

El **aprendizaje por transferencia** consiste en reutilizar un modelo previamente entrenado en una tarea específica como punto de partida para resolver otra tarea. Esta técnica aprovecha el conocimiento (patrones o características) aprendido por el modelo en la tarea original para mejorar el rendimiento, acelerar el proceso de entrenamiento, o requerir menos datos de entrenamiento en la nueva tarea.

A continuación, vamos a ver un ejemplo de aprendizaje por transferencia utilizando la librería [TensorFlow](https://www.tensorflow.org/). En particular, utilizaremos una red pre-entrenada en ImageNet como extractor de características para resolver un problema de clasificación multi-clase.

Antes de empezar, vamos a utilizar el método [set_random_seed()](https://www.tensorflow.org/api_docs/python/tf/keras/utils/set_random_seed) para establecer el valor de la **semilla** y garantizar la reproducibilidad de los resultados.

In [1]:
# Fijar la semilla
from tensorflow.keras.utils import set_random_seed

seed = 121
set_random_seed(seed)  # establece todas las semillas aleatorias del programa (Python, NumPy y TensorFlow)

<hr>

### **1. Conjunto de datos**

En esta práctica vamos a utilizar un conjunto para clasificación de imágenes denominado **Flowers recognition**, adaptado de la versión disponible en [Kaggle](https://www.kaggle.com/alxmamaev/flowers-recognition). Para ello, es necesario descargar el fichero [`flowers.zip`](https://drive.google.com/file/d/1ue8RLW1vS6Kz_8pVjhHVXtRYi-VIXaxV/view?usp=sharing) que contiene:

*   Una carpeta `images` con 4315 imágenes de flores correspondientes a cinco clases (*daisy*, *dandelion*, *rose*, *sunflower*, *tulip*).
*   Un fichero `labels.csv` con los nombres de todas las imágenes (columna *image_name*) y la clase a la que pertenece cada imagen (columna *class*).

El siguiente paso consiste en leer el fichero CSV utilizando la función [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), disponible en la libreria `pandas`.

In [2]:
# Descargar el conjunto de datos Flowers y leer el fichero CSV
import pandas as pd
from google.colab import drive

# Montar el Google Drive en el directorio del proyecto y descomprimir el fichero con los datos
drive.mount('/content/gdrive')
!unzip -n '/content/gdrive/My Drive/datasets/flowers.zip' >> /dev/null  # ACTUALIZAR: ruta al fichero comprimido

# Especificar las rutas al directorio con las imágenes y al fichero con las etiquetas
data_path = 'flowers/'
img_dir = data_path + 'images/'
csv_file = data_path + 'labels.csv'

# Leer el fichero CSV con las etiquetas
x_col = 'image_name'
y_col = 'class'
df = pd.read_csv(csv_file, dtype = {y_col: 'category'})

Mounted at /content/gdrive
unzip:  cannot find or open /content/gdrive/My Drive/datasets/flowers.zip, /content/gdrive/My Drive/datasets/flowers.zip.zip or /content/gdrive/My Drive/datasets/flowers.zip.ZIP.


FileNotFoundError: [Errno 2] No such file or directory: 'flowers/labels.csv'

A continuación, vamos a dividir los datos en tres particiones: **entrenamiento, validación y test**. Para ello utilizaremos el método [train_test_split()](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), disponible en la librería `scikit-learn`, y las siguientes proporciones: 70% de las imágenes para entrenamiento, 15% para validación y 15% para test.

In [ ]:
# Dividir el conjunto en entrenamiento, validación y test (70:15:15)
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42, stratify=df[y_col])
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp[y_col])

print(f'Número de ejemplos del conjunto de entrenamiento: {df_train.shape[0]}')
print(f'Número de ejemplos del conjunto de validación: {df_val.shape[0]}')
print(f'Número de ejemplos del conjunto de test: {df_test.shape[0]}')

Por último, vamos a normalizar los datos de entrada y generar los *batches* necesarios para entrenar la red que se definirá más adelante. Para ello definiremos, en primer lugar, una función `load_and_preprocess_image(image_name, label_str)` que permite cargar y preprocesar las imágenes. Revisa en detalle lo que hace cada una de las funciones utilizadas en este proceso.

In [ ]:
import tensorflow as tf

# Especificar información dependiente del conjunto de datos
img_width = img_height = 224  # dimensiones de la imagen
n_channels = 3                # número de canales (RGB)
n_classes = 5                 # número de clases

# Cargar y preprocesar imágenes
def load_and_preprocess_image(image_name, label_str):
    image_path = tf.strings.join([img_dir, image_name])
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=n_channels)
    image = tf.image.resize(image, [img_width, img_height])
    image = image / 255.0                               # normalizar
    label = tf.strings.to_number(label_str, tf.int32)   # conversión de etiquetas a enteros
    label = tf.one_hot(label, depth=n_classes)          # codificación one-hot
    return image, label

A continuación, definiremos una función `get_dataset(df)`, la cual recibirá una variable de tipo `DataFrame` y generará el conjunto de datos correspondiente, utilizando el método [`from_tensor_slices()`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#from_tensor_slices) de la clase [`Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset). En esta misma función podemos, una vez definido el `Dataset`, utilizaremos el método [`map()`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#map) para aplicar la función `load_and_preprocess_image()` anteriormente definida.

**NOTA:** Puede ser útil usar la propiedad [DataFrame.values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.values.html), que devuelve una representación Numpy del DataFrame.

In [ ]:
# Función para crear conjunto de datos
from tensorflow.data import Dataset

def get_dataset(df):
    image_paths = df[x_col].values
    labels = df[y_col].values
    dataset = Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(load_and_preprocess_image)
    return dataset

Por último, utilizaremos la función previamente definida, `get_dataset()`, para crear los conjuntos de datos de las tres particiones (entrenamiento, validación y test). También crearemos los batches utilizando el método [`batch()`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#batch) de la clase `Dataset` y un tamaño de 128.

In [ ]:
# Crear los conjuntos de datos y preparar los lotes (batches)
batch_size = 128
train_dataset = get_dataset(df_train).batch(batch_size)
val_dataset = get_dataset(df_val).batch(batch_size)
test_dataset = get_dataset(df_test).batch(batch_size)

print(f'Número de lotes del conjunto de entrenamiento: {len(train_dataset)}')
print(f'Número de lotes del conjunto de validación: {len(val_dataset)}')
print(f'Número de lotes del conjunto de test: {len(test_dataset)}')

print(f'\nBatches del conjunto de entrenamiento:')
for image_batch, label_batch in train_dataset.take(1):
    print(f"\tImage batch shape: {image_batch.shape}")
    print(f"\tLabel batch shape: {label_batch.shape}")

<hr>

### **2. Red convolucional**

Una alternativa a crear una CNN y entrenarla desde cero es utilizar modelos pre-entrenados con grandes conjuntos de datos. En esta práctica, vamos a utilizar el modelo [VGG16](https://www.tensorflow.org/api_docs/python/tf/keras/applications/vgg16) pre-entrenado con [ImageNet](https://www.image-net.org/), una base de datos con más de 14 millones de imágenes correspondientes a más de 20 mil clases. En la librería `TensorFlow` puedes encontrar otros [modelos pre-entrenados](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

Para ello, vamos a definir una función `get_model()` en la que cargaremos la base convolucional del modelo utilizando el método [VGG16()](https://www.tensorflow.org/api_docs/python/tf/keras/applications/VGG16) con los siguientes parámetros:
* `include_top=False`: utiliza solamente la base convolucional, eliminando las capas completamente conectadas.
* `weights='imagenet'`: carga los pesos de la red pre-entrada con ImageNet.

A continuación, congelaremos los pesos de la base convolucional para que no se modifiquen durante la fase de entrenamiento. De esta forma, la red VGG16 se utilizará como extractor de características. Para ello, será necesario modificar el atributo `trainable` de las capas:
* `layer.trainable = False`: por defecto, su valor es `True`.

El siguiente paso consiste en añadir una capa que transforme los mapas de características obtenidos con la VGG16 en un vector y, finalmente, añadir capas completamente conectadas hasta obtener la salida. Para ello, utilizaremos las siguientes capas:
* Una capa de agrupamiento global (GAP) que aplique la función promedio.
* Dos capas complementamente conectadas: una de tamaño 256 y otra que será la capa de salida, que permitirá resolver el problema de clasificación multi-clase (5 clases).

Por último, crearemos el modelo utilizando un modelo utilizando el método [Model()](https://www.tensorflow.org/api_docs/python/tf/keras/Model), que recibirá como entrada la base convolucional de la VGG16 y como salida la última capa completamente conectada del modelo.

**NOTA:** Al no haber creado un [modelo secuencial](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential), cada capa debe recibir como entrada la salida de la capa anterior. Código de ejemplo:

```python
x = base_model.output
x = GlobalAveragePooling2D()(x)
```

In [ ]:
# Crear el modelo (base convolucional VGG16 + GAP + capa oculta de 256 neuronas + capa de salida)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

def get_model():

  # Cargar la base convolucional del modelo VGG16 pre-entrenado en ImageNet
  base_model = VGG16(weights='imagenet', include_top=False, input_shape=(img_width,img_height,3))

  # Ajustar los parámetros de las nuevas capas del modelo, dejando fijos los parámetros del resto de capas
  for layer in base_model.layers:
      layer.trainable = False   # por defecto, layer.trainable es True

  # Añadir nuevas capas a continuación de la base convolucional, para resolver la tarea de aprendizaje
  x = base_model.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(256, activation='relu')(x)

  # Añadir una última capa completamente conectada con 5 neuronas (número de clases) para obtener la salida de la red
  predictions = Dense(n_classes, activation='softmax')(x)

  # Crear el modelo final e imprimir su representacion en modo texto
  model = Model(inputs=base_model.input, outputs=predictions)

  return model

Por último, vamos a crear el modelo invocando la función anterior y a utilizar el método [`summary()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#summary) para imprimir una representación en modo texto de la arquitectura definida. Con este método es posible visualizar también el número de parámetros de cada capa de la red.

In [ ]:
# Crear el modelo e imprimir su representación en modo texto
model = get_model()
model.summary()

<hr>

### **3. Entrenamiento**

Una vez definida la arquitectura de la CNN (la base convolucional de VGG16 + GAP + una capa oculta de 256 neuronas + la capa de salida), el siguiente paso es configurar el modelo para el entrenamiento. Para ello utilizaremos el método [`compile()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile), siendo estos algunos de sus parámetros más relevantes:

* `optimizer`: nombre del optimizador (`Adam`, `RMSProp`, etc.) y tasa de aprendizaje (`learning_rate`). En la web de `TensorFlow` puedes encontrar otros [optimizadores](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers).
* `loss`: función de pérdida (`mean_squared_error`, `binary_crossentropy`, `categorical_crossentropy`, etc.). En la web de `TensorFlow` puedes encontrar otras [funciones de pérdida](https://www.tensorflow.org/api_docs/python/tf/keras/losses).
* `metrics`: métricas que se evalúan para los datos de entrenamiento y validación (`accuracy`, etc.). En la web de `TensorFlow` puedes encontrar otras [métricas](https://www.tensorflow.org/api_docs/python/tf/keras/metrics).

Para configurar el proceso de aprendizaje, utiliza el optimizador [Adam](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam), una tasa de aprendizaje de `0.001` y una función de pérdida apropiada, teniendo en cuenta la tarea a resolver.

In [ ]:
# Configurar el proceso de aprendizaje
from tensorflow.keras.optimizers import Adam

opt = Adam(learning_rate=0.001)                 # optimizador Adam
model.compile(loss='categorical_crossentropy',  # función de pérdida para problemas de clasificación multi-clase
              optimizer=opt,
              metrics=['accuracy'])

A continuación, vamos a entrenar el modelo para buscar los parámetros que hagan mínima la función de pérdida. Para ello utilizaremos el método [`fit()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit), que necesita que le suministremos los datos de entrenamiento y validación, y el número de *epochs*.

In [ ]:
# Entrenar el modelo
history = model.fit(train_dataset,
          epochs=6,   # número de epochs
          verbose=2,  # muestra información al finalizar cada epoch
          validation_data=val_dataset)

Por último, vamos a imprimir el error mínimo de entrenamiento y validación accediendo al atributo `history`.

In [ ]:
# Imprimir el error mínimo (entrenamiento y validación)
import numpy as np

train_trace = np.array(history.history['loss'])
print(f'\nError mínimo en entrenamiento: {min(train_trace):.6f}')

val_trace = np.array(history.history['val_loss'])
print(f'Error mínimo en validación: {min(val_trace):.6f}')

### **4. Evaluación**

Hemos visto cómo crear y entrenar una CNN a partir de un modelo pre-entrenado, utilizando una configuración de hiperparámetros que no necesariamente es la mejor. Lo ideal sería realizar una búsqueda de hiperparámetros y, una vez obtenida la mejor configuración, evaluar el modelo sobre el conjunto de test y así obtener el resultado final.

A continuación, se muestra el código para evaluar el modelo final en el conjunto de test utilizando el método [`evaluate()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#evaluate).

In [ ]:
# Evaluar el modelo en el conjunto de test
test_loss, test_acc = model.evaluate(test_dataset, verbose=1)
print("test_loss: %.4f, test_acc: %.4f" % (test_loss, test_acc))

Por último, además de analizar el error obtenido, podemos hacer predicciones con el modelo entrenado. Para ello utilizaremos el método [`predict()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#predict), al que le suministraremos los datos sobre los que realizar las predicciones (en este caso, los datos de test).

In [ ]:
# Obtener las predicciones para todos los ejemplos del conjunto de test y mostrar los valores obtenidos para las N primeras imágenes
predictions = model.predict(test_dataset, verbose=1)
n = 2

for i in range(0,n):
  print("\n Ejemplo", i)
  print("\t Probabilidades para las 5 clases:", predictions[i])
  print("\t Clase predicha: %i, Probabilidad: %.4f" % (np.argmax([predictions[i]]), np.max(predictions[i])))

### **5. Visualización**

Las redes de aprendizaje profundo en general, y las CNNs en particular, se consideran modelos de caja negra porque, aunque son capaces de hacer predicciones con una alta precisión, no está claro cómo lo hacen.

En este contexto puede resultar útil visualizar:
*   Los **filtros de las capas convolucionales**, para descubrir los tipos de características o patrones que el modelo es capaz de detectar.
*   Los **mapas de características** obtenidos pora las capas convolucionales del modelo, para entender qué características se detectan en una imagen de entrada dada.


**FILTROS CONVOLUCIONALES**

Cada capa convolucional tiene dos conjuntos de pesos: los valores de los filtros y los vectores de sesgo (si se utilizan, `use_bias=True`).

Partiendo del modelo previamente entrenado, basado en la red VGG16 pre-entrenada en ImageNet, vamos a ver cómo visualizar los filtros aprendidos. En concreto, nos centraremos en la primera capa convolucional del modelo que está compuesta de 64 filtros de tamaño 3x3x3 y visualizaremos los 10 primeros filtros, representando cada uno de sus tres canales de forma individual.

Para ello, vamos a empezar por recuperar los pesos de la primera capa convolucional utilizando el método [get_weights()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Layer#get_weights), disponible en `TensorFlow`.

In [ ]:
# Recuperar los pesos de la primera capa convolucional
filters, biases = model.layers[1].get_weights()

A continuación, normalizaremos los valores de los filtros al intervalor [0, 1] para poder visualizarlos correctamente.

In [ ]:
# Normalizar los valores de los filtros al intervalo [0, 1]
f_min = filters.min()
f_max = filters.max()
filters = (filters - f_min) / (f_max - f_min)

Por último, utilizaremos la interfaz [`pyplot`](https://matplotlib.org/3.5.3/api/_as_gen/matplotlib.pyplot.html) de la librería `matplotlib` para representar gráficamente los 10 primeros filtros de la primer capa convolucional. En particular, representaremos individualmente cada uno de los tres canales de cada filtro, obteniendo una representación 10x3.

**NOTA:** Algunas de las funciones útiles de `matplotlib` son [`subplot()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplot.html), [`imshow()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) y [`show()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.show.html).

In [ ]:
# Representar los 10 primeros filtros (cada canal por separado)
from matplotlib import pyplot

n_filters = 10
n_channels = 3
ix = 1

for i in range(n_filters):
  f = filters[:, :, :, i]
  for j in range(n_channels):
    ax = pyplot.subplot(n_filters, n_channels, ix)
    ax.set_xticks([])
    ax.set_yticks([])
    pyplot.imshow(f[:, :, j], cmap='gray')
    ix += 1

# Mostrar la figura
pyplot.show()

**MAPAS DE CARACTERÍSTICAS**

Los mapas características capturan el resultado de aplicar los filtros de las capas convolucionales a los datos de entrada, que pueden ser una imagen u otros mapas de características. En general, los mapas de características de las primeras capas capturan características de bajo nivel mientras que los mapas de las últimas capas suelen ser más específicos del problema a resolver.

Partiendo del modelo previamente entrenado, basado en la red VGG16 pre-entrenada en ImageNet, vamos a ver cómo visualizar los 64 mapas de características de tamaño 224x224 obtenidos con la primera capa convolucional del modelo. Para ello, utilizaremos el método [Model()](https://www.tensorflow.org/api_docs/python/tf/keras/Model), disponible en TensorFlow, para redefinir el modelo de salida justo después de la primera capa convolucional.

In [ ]:
# Redefinir el modelo de salida justo después de la primera capa convolucional
model_v = Model(inputs=model.inputs, outputs=model.layers[1].output)
model_v.summary()

A continuación, cargaremos una imagen de ejemplo del conjunto de datos utilizado en esta práctica.

In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Cargar una imagen de ejemplo del conjunto de datos
img = load_img('flowers/images/100080576_f52e8ee070_n.jpg', target_size=(224, 224))
img = img_to_array(img)
img = np.expand_dims(img, axis=0)    # muestra
img = preprocess_input(img)          # imagen pre-procesada (VGG16)

El siguiente paso consiste en obtener los mapas de características de la imagen de entrada, utilizando el método [`predict()`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#predict).

In [ ]:
# Obtener los mapas de características a partir de la imagen de ejemplo
feature_maps = model_v.predict(img)

Por último, utilizaremos la interfaz pyplot de la librería matplotlib para representar gráficamente los 64 mapas de características en formato 8x8.

**NOTA:** Algunas de las funciones útiles de matplotlib son subplot(), imshow() y show().

In [ ]:
# Representar los 64 mapas de características, en formato 8x8
square = 8
ix = 1
for i in range(square):
  for j in range(square):
    ax = pyplot.subplot(square, square, ix)
    ax.set_xticks([])
    ax.set_yticks([])
    pyplot.imshow(feature_maps[0, :, :, ix-1], cmap='gray')
    ix += 1

# Mostrar la figura
pyplot.show()

### **6. Ejercicios**

A continuación, se proponen una serie de ejercicios para su resolución.

**EJERCICIO 1**

La primera capa convolucional del modelo definido anteriormente está compuesta de 64 filtros de tamaño 3x3x3. En el ejemplo de visualización de filtros convolucionales hemos representado cada canal de forma individual pero, dado que son tres, es posible visualizarlos de manera conjunta como si de una imagen en color se tratara.

Teniendo en cuenta esto, visualiza los 64 filtros de la primera capa convolucional en un formato 8x8 (similar al que hemos utilizado para visualizar los mapas de características).

**NOTA:**  Normaliza los valores de los filtros al intervalo [0, 1] para poder visualizarlos correctamente.

In [ ]:
# To-Do: Solución al ejercicio 1

**EJERCICIO 2**

Partiendo del modelo original, sustituye la red VGG16 por una [InceptionV3](https://www.tensorflow.org/api_docs/python/tf/keras/applications/inception_v3), también pre-entrenada en ImageNet.

¿Cómo afecta esta modificación al número de capas del modelo y al número de parámetros que se ajustan durante el entrenamiento? En vista de la arquitectura obtenida, ¿consideras necesario hacer algún ajuste en las capas finales del modelo (clasificación)?

In [ ]:
# To-Do: Solución al ejercicio 2